# Additional End of week Exercise - week 2

In [7]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from artists import tools, handle_tool_calls
from artists_met import get_image

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if not(openai_api_key):
    print("OpenAI API Key not set")

LAMA_MODEL = "llama3.2"
DEEPSEEK_MODEL = "deepseek-r1:1.5b"
GPT_MODEL = "gpt-4.1-mini"
OLLAMA_BASE_URL = "http://localhost:11434"

openai = OpenAI()
hasOllama = requests.get(OLLAMA_BASE_URL).content
print(f"Ollama is running: {hasOllama}")

ollama = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key='ollama')

Ollama is running: b'Ollama is running'


In [3]:
system_message = """You are a helpful assistant, working for the Metropolitan Museum of Art in New York. You provide information about artists and shows images of their artworks. 
The tools that are provided to you in this chat give you access to a dataset with information about artists, including their names, the period in which he/she lived, and the collections their work belongs to. Also from an artist's name, you can get a random artwork from that artist, including the title and an image of the artwork.
If you don't find an artist by the name the user provided, first try to use the get_artist_suggestions to check if you can ask if the user meant one of the suggested artists. If you find a suggestion that matches the user's intent, you can ask the user if they meant that artist. If the user confirms, you can then use the get_random_artwork_from_artist tool to get information about that artist and their artworks.
Keep in mind that the user does not always asks for an artist but also can start a normal social conversation, like a greeting or friendly talk, etcera.
From the collections, you can infer the style and period of the artists' works.
When asked about an artist you will provide relevant information based on the dataset. 
If you don't have information about a specific artist or collection, you will politely inform the user that you don't have that information.
When appropriate, you can suggest an artist from the dataset that matches the user's interests,or that of a similar style or period. You can also suggest a random artist by using the tools, when you think that's appropriate, but explain that it is a different style or period.
Only stick to the artists and the information about them you can get by using the tools! Don't use other information you have in your internal dataset. Do not make up any information, if you don't know, say you don't know.
"""

In [ ]:
def chat_GPT(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=GPT_MODEL, messages=messages, tools=tools)
    image_url = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, image_url, artist_name, title = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=GPT_MODEL, messages=messages, tools=tools)
    
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    default_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/7/73/The_Metropolitan_Museum_of_Art_Logo.svg/250px-The_Metropolitan_Museum_of_Art_Logo.svg.png"
    image = get_image(default_image_url)
    if image_url:
        image = get_image(image_url)

    return history, image, # artist_name, title

In [5]:
def chat_ollama(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = ollama.chat.completions.create(model=DEEPSEEK_MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, image_url, artist_name, title = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=DEEPSEEK_MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}] 
       
    image = None
    if image_url:
        image = get_image(image_url)    
    
    return history, image, # artist_name, title

In [22]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat_GPT, inputs=chatbot, outputs=[chatbot, image_output]
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\Joop.Rosier\ai-engineer-udemy-projects\llm_engineering_joop\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Joop.Rosier\ai-engineer-udemy-projects\llm_engineering_joop\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Joop.Rosier\ai-engineer-udemy-projects\llm_engineering_joop\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Joop.Rosier\ai-engineer-udemy-projects\llm_engineering_joop\.venv\Lib\site-packages\gradio\blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^

In [123]:
from urllib.parse import quote
quoted_name = quote("Vincent van Gogh")
print(quoted_name)

Vincent%20van%20Gogh
